# Exploratory Data Analysis - INX Future Inc Employee Performance

## Objective
Conduct comprehensive exploratory analysis to understand:
1. Distribution of performance across departments
2. Key factors influencing employee performance
3. Relationships between variables
4. Insights for business recommendations

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

print("Libraries loaded successfully!")

## 1. Load Processed Data

In [ ]:
# Load processed data
df = pd.read_csv('../../data/processed/employee_data_processed.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
display(df.head())

## 2. Target Variable Analysis - Performance Rating

In [ ]:
# Identify the target variable (Performance Rating)
# Common column names: PerformanceRating, Performance, EmpPerformanceRating, etc.
target_col = [col for col in df.columns if 'performance' in col.lower() and 'rating' in col.lower()]

if target_col:
    target = target_col[0]
    print(f"Target Variable: {target}")
else:
    # If not found, try alternative names
    target = 'PerformanceRating'  # Default assumption
    print(f"Using assumed target: {target}")

print(f"\nPerformance Rating Distribution:")
print(df[target].value_counts().sort_index())
print(f"\nPerformance Rating Statistics:")
print(df[target].describe())

In [ ]:
# Visualize performance distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Count plot
performance_counts = df[target].value_counts().sort_index()
axes[0].bar(performance_counts.index, performance_counts.values, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Performance Rating', fontsize=12)
axes[0].set_ylabel('Number of Employees', fontsize=12)
axes[0].set_title('Distribution of Performance Ratings', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(performance_counts.values):
    axes[0].text(performance_counts.index[i], v + 5, str(v), ha='center', fontweight='bold')

# Percentage plot
performance_pct = (df[target].value_counts(normalize=True).sort_index() * 100).round(1)
axes[1].pie(performance_pct.values, labels=performance_pct.index, autopct='%1.1f%%',
            startangle=90, colors=sns.color_palette('pastel'))
axes[1].set_title('Performance Rating Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../../data/processed/performance_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Check for class imbalance
if performance_pct.max() > 50:
    print("⚠️ Warning: Class imbalance detected. Consider using SMOTE or class weights in modeling.")

## 3. Department-wise Performance Analysis

In [ ]:
# Identify department column
dept_col = [col for col in df.columns if 'department' in col.lower()]
if dept_col:
    dept_col = dept_col[0]
else:
    dept_col = 'EmpDepartment'  # Default assumption

print(f"Department Column: {dept_col}")
print(f"\nDepartments: {df[dept_col].unique()}")
print(f"\nEmployee Distribution by Department:")
print(df[dept_col].value_counts())

In [ ]:
# Department-wise performance analysis
dept_performance = df.groupby(dept_col)[target].agg(['mean', 'median', 'std', 'count']).round(2)
dept_performance = dept_performance.sort_values('mean', ascending=False)

print("\n" + "="*80)
print("DEPARTMENT-WISE PERFORMANCE ANALYSIS")
print("="*80)
display(dept_performance)

# Identify best and worst performing departments
best_dept = dept_performance.index[0]
worst_dept = dept_performance.index[-1]

print(f"\n📊 Key Findings:")
print(f"  ✓ Best Performing: {best_dept} (Avg: {dept_performance.loc[best_dept, 'mean']})")
print(f"  ✗ Needs Improvement: {worst_dept} (Avg: {dept_performance.loc[worst_dept, 'mean']})")
print(f"  📈 Performance Gap: {(dept_performance.loc[best_dept, 'mean'] - dept_performance.loc[worst_dept, 'mean']):.2f}")

In [ ]:
# Visualize department performance
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Average performance by department
dept_performance['mean'].plot(kind='barh', ax=axes[0,0], color='steelblue', edgecolor='black')
axes[0,0].set_xlabel('Average Performance Rating', fontsize=11)
axes[0,0].set_ylabel('Department', fontsize=11)
axes[0,0].set_title('Average Performance Rating by Department', fontsize=13, fontweight='bold')
axes[0,0].axvline(x=df[target].mean(), color='red', linestyle='--', label=f'Overall Avg: {df[target].mean():.2f}')
axes[0,0].legend()
axes[0,0].grid(axis='x', alpha=0.3)

# 2. Employee count by department
dept_performance['count'].plot(kind='barh', ax=axes[0,1], color='coral', edgecolor='black')
axes[0,1].set_xlabel('Number of Employees', fontsize=11)
axes[0,1].set_ylabel('Department', fontsize=11)
axes[0,1].set_title('Employee Count by Department', fontsize=13, fontweight='bold')
axes[0,1].grid(axis='x', alpha=0.3)

# 3. Performance distribution by department (box plot)
df_sorted = df.sort_values(dept_col)
sns.boxplot(data=df_sorted, y=dept_col, x=target, ax=axes[1,0], palette='Set2')
axes[1,0].set_xlabel('Performance Rating', fontsize=11)
axes[1,0].set_ylabel('Department', fontsize=11)
axes[1,0].set_title('Performance Distribution by Department', fontsize=13, fontweight='bold')
axes[1,0].grid(axis='x', alpha=0.3)

# 4. Performance rating composition by department (stacked bar)
dept_perf_crosstab = pd.crosstab(df[dept_col], df[target], normalize='index') * 100
dept_perf_crosstab.plot(kind='barh', stacked=True, ax=axes[1,1], 
                        colormap='viridis', edgecolor='white', linewidth=0.5)
axes[1,1].set_xlabel('Percentage (%)', fontsize=11)
axes[1,1].set_ylabel('Department', fontsize=11)
axes[1,1].set_title('Performance Rating Composition by Department (%)', fontsize=13, fontweight='bold')
axes[1,1].legend(title='Rating', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('../../data/processed/department_performance_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical test - ANOVA for department differences
from scipy.stats import f_oneway

dept_groups = [group[target].values for name, group in df.groupby(dept_col)]
f_stat, p_value = f_oneway(*dept_groups)

print("\nANOVA Test - Department Performance Differences")
print("="*80)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n✓ Result: Significant differences exist between departments (p < 0.05)")
else:
    print("\n✗ Result: No significant differences between departments (p >= 0.05)")

## 4. Feature Importance Analysis - Correlation with Performance

In [ ]:
# Select numerical features
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove ID column and target variable
numerical_features = [col for col in numerical_features if col not in ['EmpNumber', target]]

print(f"Numerical Features ({len(numerical_features)}): {numerical_features}")

In [ ]:
# Calculate correlation with target variable
correlations = df[numerical_features + [target]].corr()[target].drop(target).sort_values(ascending=False)

print("\n" + "="*80)
print("CORRELATION WITH PERFORMANCE RATING")
print("="*80)
print(correlations)

# Identify top positive and negative correlations
print(f"\n📊 Top 5 Positive Correlations:")
for i, (feat, corr) in enumerate(correlations.head(5).items(), 1):
    print(f"  {i}. {feat}: {corr:.3f}")

print(f"\n📊 Top 5 Negative Correlations:")
for i, (feat, corr) in enumerate(correlations.tail(5).items(), 1):
    print(f"  {i}. {feat}: {corr:.3f}")

In [ ]:
# Visualize correlations
fig, ax = plt.subplots(figsize=(12, 8))

# Sort by absolute correlation
top_corr = correlations.abs().sort_values(ascending=True).tail(15)
colors = ['red' if correlations[feat] < 0 else 'green' for feat in top_corr.index]

top_corr_values = [correlations[feat] for feat in top_corr.index]
ax.barh(range(len(top_corr)), top_corr_values, color=colors, edgecolor='black', alpha=0.7)
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr.index, fontsize=10)
ax.set_xlabel('Correlation with Performance Rating', fontsize=12)
ax.set_title('Top 15 Features by Correlation with Performance', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(top_corr_values):
    ax.text(v + 0.01 if v > 0 else v - 0.01, i, f'{v:.3f}', 
            va='center', ha='left' if v > 0 else 'right', fontweight='bold')

plt.tight_layout()
plt.savefig('../../data/processed/feature_correlations.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Correlation Matrix - Feature Relationships

In [ ]:
# Select top features for correlation matrix
top_features = correlations.abs().sort_values(ascending=False).head(10).index.tolist()
correlation_matrix = df[top_features + [target]].corr()

# Visualize correlation matrix
plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Top Features vs Performance', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../../data/processed/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Categorical Features Analysis

In [ ]:
# Select categorical features
categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove department as it's already analyzed
categorical_features = [col for col in categorical_features if col != dept_col]

print(f"Categorical Features ({len(categorical_features)}): {categorical_features}")

# Analyze impact on performance
print("\n" + "="*80)
print("CATEGORICAL FEATURES - IMPACT ON PERFORMANCE")
print("="*80)

for feat in categorical_features[:5]:  # Analyze top 5
    if df[feat].nunique() <= 10:  # Only for features with reasonable number of categories
        print(f"\n{feat}:")
        perf_by_cat = df.groupby(feat)[target].agg(['mean', 'count']).round(2)
        display(perf_by_cat.sort_values('mean', ascending=False))

In [ ]:
# Visualize key categorical features
key_categorical = [col for col in categorical_features if df[col].nunique() <= 8][:4]

if len(key_categorical) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, feat in enumerate(key_categorical):
        if idx < 4:
            perf_by_cat = df.groupby(feat)[target].mean().sort_values(ascending=False)
            perf_by_cat.plot(kind='bar', ax=axes[idx], color='teal', edgecolor='black', alpha=0.7)
            axes[idx].set_title(f'Average Performance by {feat}', fontsize=12, fontweight='bold')
            axes[idx].set_ylabel('Average Performance Rating', fontsize=10)
            axes[idx].set_xlabel(feat, fontsize=10)
            axes[idx].axhline(y=df[target].mean(), color='red', linestyle='--', 
                            label=f'Overall Avg: {df[target].mean():.2f}')
            axes[idx].legend()
            axes[idx].grid(axis='y', alpha=0.3)
            axes[idx].tick_params(axis='x', rotation=45)
    
    # Hide empty subplots
    for idx in range(len(key_categorical), 4):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('../../data/processed/categorical_features_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

## 7. Key Insights Summary

In [ ]:
# Generate comprehensive insights summary
print("\n" + "="*100)
print(" " * 30 + "EXPLORATORY DATA ANALYSIS SUMMARY")
print("="*100)

print("\n1. OVERALL PERFORMANCE METRICS")
print("-" * 100)
print(f"   • Total Employees: {len(df):,}")
print(f"   • Average Performance Rating: {df[target].mean():.2f}")
print(f"   • Performance Range: {df[target].min()} to {df[target].max()}")
print(f"   • Standard Deviation: {df[target].std():.2f}")

print("\n2. DEPARTMENT PERFORMANCE")
print("-" * 100)
print(f"   • Best Performing Department: {best_dept} (Avg: {dept_performance.loc[best_dept, 'mean']:.2f})")
print(f"   • Department Needing Attention: {worst_dept} (Avg: {dept_performance.loc[worst_dept, 'mean']:.2f})")
print(f"   • Performance Gap: {(dept_performance.loc[best_dept, 'mean'] - dept_performance.loc[worst_dept, 'mean']):.2f} points")
print(f"   • Statistical Significance: {'Yes' if p_value < 0.05 else 'No'} (p={p_value:.4f})")

print("\n3. TOP 3 FACTORS AFFECTING PERFORMANCE")
print("-" * 100)
top_3_factors = correlations.abs().sort_values(ascending=False).head(3)
for i, (factor, corr_val) in enumerate(top_3_factors.items(), 1):
    direction = "positive" if correlations[factor] > 0 else "negative"
    print(f"   {i}. {factor}")
    print(f"      - Correlation: {correlations[factor]:.3f} ({direction} relationship)")
    print(f"      - Impact: {'High' if abs(correlations[factor]) > 0.5 else 'Moderate' if abs(correlations[factor]) > 0.3 else 'Low'}")

print("\n4. CLASS BALANCE")
print("-" * 100)
for rating, pct in performance_pct.items():
    print(f"   • Rating {rating}: {pct:.1f}% ({df[target].value_counts().sort_index()[rating]:,} employees)")

print("\n5. KEY RECOMMENDATIONS FOR MODELING")
print("-" * 100)
if performance_pct.max() > 50:
    print("   • Address class imbalance using SMOTE or class weights")
print(f"   • Focus on top {len(top_3_factors)} features for initial modeling")
print("   • Consider department-specific models if performance gaps are significant")
print("   • Include interaction terms between highly correlated features")

print("\n" + "="*100)
print("Analysis complete! Ready for model training.")
print("="*100)

## 8. Export Analysis Results

In [ ]:
# Export key findings to CSV
dept_performance.to_csv('../../data/processed/department_performance_summary.csv')
correlations.to_csv('../../data/processed/feature_correlations.csv')

print("Analysis results exported successfully!")
print("  - department_performance_summary.csv")
print("  - feature_correlations.csv")